# CE Evolution

The common-envelope (CE) evolution is one of the most uncertain phases of binary evolution. It is triggered when mass transfer becomes unstable, causing the binary to become engulfed in the donor’s envelope. Inside the envelope, drag forces transfer orbital energy and angular momentum from the binary orbit to the surrounding envelope material. If the orbital energy released by the orbital shrinkage is sufficient, the envelope is ejected and the binary survives the CE phase. Otherwise the binary merges within the CE.

The physics governing CE evolution is not well understood. Even 3‑D hydrodynamic simulations struggle to model the CE because of the many orders of magnitude in the relevant spatial and temporal scales required to acurately model this phase, as well as the computational cost associated with resolving them.

Binary population synthesis codes like POSYDON therefore rely on simplified prescriptions to determine the outcome of CE evolution. These prescriptions introduce additional free parameters that are very poorly constrained.

But before we dive in, let’s see how POSYDON determines whether mass transfer becomes unstable, which leads binaries to enter a CE.


# Onset of CE

In most cases, unstable mass transfer occurs when a massive donor with a less massive accretor starts filling its Roche lobe (RL). Rapid codes use a critical mass ratio $q_{crit}$, bellow which binaries are consider to enter unstable mass transfer. In POSYDON, we assess mass-transfer stability for every binary in our grids. If any of these conditions are met, the MESA run is terminated and the binary is flagged as undergoing unstable mass transfer.

## Stability criteria 
1. The mass rate becomes larger than $0.1 \, M_\odot \, yr^{-1}$
2. If the star radius is larger than the second Lagrangian point $L_2$
3. If both stars overfill their RL while on of the is post-MS star
4. For a CO companion : The photon trapping radius (Begelman 1979; King & Begelman 1999), reaches the RL radius of the accretor.
   


# The α<sub>CE</sub>–λ Prescription

In this lab, we will go through the **α<sub>CE</sub>–λ formalism** by calculating both the binding energy and the post-CE separation of the binary.

---

### Motivation
---
The **post-common-envelope (post-CE) outcome** of a binary system is very sensitive to how we define the boundary between the core and the envelope of the donor star, as well as the free parameter $\alpha$


In traditional binary population synthesis codes, the **λ values** are calculated by fitting formulas based on a pre-computed grid of stellar tracks. Binary interactions, often change the evolution of stellar interior and therefore the  **λ values**. POSYDON, however has access to the final profile of the stars before the enter the CE and is able calculate the **λ parameter**. 




Here, we aim to calculate **binding energy** and **λ values** for **our own definition of the core boundary**. This allows us to explore how different choices affect the envelope ejection and the resulting post-CE orbital separation.  

---
Following the $\alpha_{CE} - \lambda$ prescription we relate the binding energy used for the ejection of the envelope with the change in orbital energy through the efficiency coeffient $a_{CE}$.

$$
    E_{bind} = a_{CE} \, \Delta E_{orb}
$$


The binding energy of the envelope is given by:  

$$
E_{\rm bind} = \int_{M(r^*)}^{M(R)} \Big( - \frac{G m}{r(m)} + \alpha_{th} U(m) \Big) dm
$$

where:
- $ r^* $ = radial coordinate of the core–envelope boundary
- $ R $ = stellar radius, with $M(R) = M_{\rm donor} $
- $ M(r^*) = M_{\rm core} $ , the remaining mass after the envelope is removed  
- $ U(m) $ = internal energy per unit mass  
- $ \alpha_{th} $ = fraction of internal energy contributing to envelope ejection  


The change in orbital energy during CE evolution is:  

$$
\Delta E_{\rm orb} = E_{\rm orb}^{f} - E_{\rm orb}^{i} 
= - \frac{G M_{\rm core} M_1}{2 a_f} + \frac{G M_{\rm donor} M_1}{2 a_0}
$$

where:
- $ M_{\rm donor} $ = donor mass  
- $ M_{\rm core} $ = post-CE core mass  
- $ M_1 $ = companion mass  
- $ a_0, a_f $ = orbital separation before and after CE  

---

In the α<sub>CE</sub>–λ scheme, the binding energy is often expressed as:  

$$
E_{\rm bind} = - \frac{G M_{\rm donor} \, M_{\rm env}}{\lambda \, R_{\rm donor}}
$$




---

### Goals of this Lab
---
In this first lab, we will:

1. We will look at a binary and the final profile of the donor and examine how changing the the core-envelope boundary and alpha parameter will affect the outcome of the CE.  
2. Explore the different option that POSYDON has for the CE
3. Run a small population using different CE prescription 
4. Compare the results of the different runs. 


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from posydon.grids.psygrid import PSyGrid
from posydon.utils import constants as const
from posydon.utils import common_functions as cf

plt.rcParams.update({
    "figure.figsize": (7, 5),
    "axes.grid": True,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.frameon": False
})

os.environ['PATH_TO_POSYDON_DATA'] = '/home/jovyan/data/POSYDON_GRIDS/'


### Loading the Grids and accessing the profiles 
---
We begin by learning how to load and use Psygrid objects. The Psygrid object is used to incorporate MESA grids into POSYDON. We can use the PsyGrid to access the POSYDON grids and pull up information about the history and the final profiles of every star and binary from our grids. 

Let's begin by loading the HMS-HMS grid.

<details>
<summary>Click for the solution</summary>

```python
from posydon.grids.psygrid import PSyGrid
grid = PSyGrid('/PATH_TO_POSYDON_data/HMS-HMS/1e+00_Zsun.h5')

````

In [ ]:
"Write you code here"
from posydon.grids.psygrid import PSyGrid
grid = PSyGrid('{PATH_TO_POSYDON}/HMS-HMS/1e+00_Zsun.h5')


Let's explore the PsyGrid object. Let's start by choosing a binary from the grid and navigating it's final profile. Choose an **<span style="color:blue">index < len(grid)</span>** and print out the final profile of the primary. 

In [ ]:
index = 7761
model = grid[index]
print(model.final_profile1)

The output is a long array. We will now utilize pandas Dataframe to make the output more readable. 
    

In [ ]:
primary   = pd.DataFrame(grid[index].final_profile1)
secondary = pd.DataFrame(grid[index].final_profile2)
binary    = pd.DataFrame(grid[index].binary_history)
print(primary)

In [ ]:
# Inspect the keys in the binary history and the star's profile
print("Primary profile keys:\n", list(primary.keys()))
print("\nBinary history keys:\n", list(binary.keys()))

As you learned in the previous Lab POSYDON has termination flags that are associated with the stability of mass transfer. For this Lab we need to select a binary that goes through unstable mass transfer during it's post MS sequence evolution.

### Exercise 1

Play around with the index in bellow and find a binary with a primary with initial mass between 17 and 50 $ M_\odot $, that goes through unstable mass transfer. You can try from this list bellow or try to find you own unique index. Binary that corresponds to the index of your choices will be inclosed by the red box. 

Indexes to choose from : ``[6768 , 4213 ,11390,  4444,  4361, 11242, 11196,  9243, 11452,  8452,  7673,  4410,
 10429, 10157, 10662,  8204,  9847,  8319,  4090, 11327, 10810, 10900,  7761,  5704,
 11325]``

In [ ]:
index = #Choose your index
binary = pd.DataFrame(grid[index].binary_history)
q = binary.star_2_mass[0]/binary.star_1_mass[0]
# similar to the 1D plotting method, we can pass a dictionary with additional plotting properties
PLOT_PROPERTIES = {
    'figsize': (6,5.),
    'log10_x' : True,
    'log10_y' : True,
    'show_fig': False,
    'close_fig': False,
    'title': f'$q={q}$',
}


grid.plot2D(
    # which parameters to plot
    x_var_str='star_1_mass',
    y_var_str='period_days',
    # z_var parameter to color the points by (only compatible with termination_flag_1)
    z_var_str=None,
    
    # what termination flag to color the points by
    termination_flag='combined_TF12',
            
    # set that the 3D grid should be used and what parameter to slice on.
    grid_3D=True, slice_3D_var_str='mass_ratio',
    # select only the models with q +/- 0.03
    # similar to q_mask above
    slice_3D_var_range=(q-0.03, q+0.03),
    **PLOT_PROPERTIES)
plt.scatter(np.log10(binary.star_1_mass[0]),np.log10(binary.period_days[0]), marker='s', facecolors='none', edgecolors='red',s=25)

### Exercise 2

Now that you selected your binary let's inspect some of it's internal properties at the onset of RLOF. In MESA the profiles correspond to zones in the star, with the zero index being the one of the outermost layer of the star. Make a plot of the specific internal energy of the star vs. its radius by accesing the keys of primary. Make an additional plot of the H mass fraction vs. radius. 

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, sharex=True, figsize=(10, 4))
"""
Write you code here 
"""
#ax1.plot(np.log10(primary.radius),primary.energy)
ax1.set_ylabel('Energy $[erg/gr]$')
ax1.set_xlabel('log($R/R_\odot$)')
#ax2.plot(np.log10(primary.radius),primary.x_mass_fraction_H)
ax2.set_xlabel('log($R/R_\odot$)')
ax2.set_ylabel('$H_1$ fraction')


<details>
<summary>Click for hint </summary>

To inspect the available keys and find how access the internal energy and radius from the profile Dataframe:
```python
    print(primary.keys())
````

<details>
<summary>Click for the solution</summary>

```python
import matplotlib.pyplot as plt 
fig, (ax1, ax2) = plt.subplots(1, 2, sharex=True, figsize=(10, 4))
ax1.plot(np.log10(primary.radius),primary.energy)
ax1.set_ylabel('Energy $[erg]$')
ax1.set_xlabel('log($R/R_\odot$)')
ax2.plot(np.log10(primary.radius),primary.x_mass_fraction_H)
ax2.set_xlabel('log($R/R_\odot$)')
ax2.set_ylabel('$H_1$ fraction')

````

### Calculating the binding energy
    From the plots we made above we see how the internal energy varies as a function of radius. We will now make a similar plot but instead of the internal energy we will plot the binding energy. The binding energy is given by the equation above and it consist of two componets: the gravitational energy and the internal energy. The internal energy is given by the final profile and represents the internal energy within each zone.

$$
E_{grav} =   \int_{M(r^*)}^{M(R)} - \frac{G m}{r} dm = \sum_{i} - \frac{G m_i}{r_i} dm_i 
$$

$$
U_{in} = \int_{M(r^*)}^{M(R)}  a_{th} U(m) dm =  \sum_{i} a_{th} U_i dm_i
$$

$$
    E_{bind} = E_{grav} + U_{in} +
$$

Let's set `common_envelope_alpha_thermal` = 1 for now. 




### Exercise 3

Fill the code bellow in the `calculate_binding_energy` function. Find the appropriate keys for the star's profile DataFrame for the radius, internal specific energy, mass and zones. 

In [ ]:
"""Write your code """
from posydon.utils import constants as const

#Function that will calculate the binding energy at every zone in the star 

def calculate_binding_energy(star,common_envelope_alpha_thermal = 1):

    Grav_energy = 0.0
    U_i = 0.0
    
    radius = np.array() 
    internal_energy = np.array() 
    mass = np.array() 
    dm = np.concatenate((-1 * np.diff(mass),[mass[-1]]))
    zones = np.array() 
    Ebind = []
    for zone_i in zones: 
        for i in range(zone_i):
            Grav_energy_i = (-const.standard_cgrav * mass[i]
                                   * const.Msun * dm[i]*const.Msun
                                   / (radius[i]*const.Rsun))
            Grav_energy = Grav_energy + Grav_energy_i
            U_i = U_i + internal_energy[i]*dm[i]*const.Msun
    
    
        Ebind.append(Grav_energy + common_envelope_alpha_thermal * U_i)
    return Ebind

<details>
<summary>Click for the solution</summary>

```python
def calculate_binding_energy(star,common_envelope_alpha_thermal = 1):

    Grav_energy = 0.0
    U_i = 0.0
    
    radius = np.array(star.radius)
    internal_energy = np.array(star.energy)
    mass = np.array(star.mass)
    dm = np.concatenate((-1 * np.diff(mass),[mass[-1]]))
    zones = primary.index
    Ebind = []
    for zone_i in zones: 
        for i in range(zone_i):
            Grav_energy_i = (-const.standard_cgrav * mass[i]
                                   * const.Msun * dm[i]*const.Msun
                                   / (radius[i]*const.Rsun))
            Grav_energy = Grav_energy + Grav_energy_i
            U_i = U_i + internal_energy[i]*dm[i]*const.Msun
    
    
        Ebind.append(Grav_energy + common_envelope_alpha_thermal * U_i)
    return Ebind

````

Now we will remake the plot and see hot the binding energy increases the deeper we go into the star. The binding energy is a negative number so we want to plot the absolute value of it  



In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, sharex=True, figsize=(10, 4))
"""
Write you code here 
"""
# 1rst subplot
ax1.set_ylabel('Binding energy $[erg]$')
ax1.set_xlabel('log($R/R_\odot$)')
# 2nd subplot
ax2.set_xlabel('log($R/R_\odot$)')
ax2.set_ylabel('$H_1$ mass fraction')

<details>
<summary>Click for the solution</summary>

```python
fig, (ax1, ax2) = plt.subplots(1, 2, sharex=True, figsize=(10, 4))

ax1.plot(np.log10(primary.radius),np.abs(calculate_binding_energy(primary)))
ax1.set_ylabel('Binding energy $[erg]$')
ax1.set_xlabel('log($R/R_\odot$)')
ax2.plot(np.log10(primary.radius),primary.x_mass_fraction_H)
ax2.set_xlabel('log($R/R_\odot$)')
ax2.set_ylabel('$H_1$ mass fraction')

````

### Exercise 4 

We have now see how the binding energy is increasing inside the star. In this exercise you will have to print out what is the binding energy for different definitions of the core-boundary at X = 0.1, 0.2 and 0.3. 


<details>
<summary>Hint</summary>

You first need to find the zone index where X_H ≤ X_H_thr. You can use function like np.where for that

In [ ]:
"""
Write the code here.

"""
X_H_1 = 0.3

E_bind_1 = 
print(f'The bindind energy for X_H = {X_H_1} is E_bind={E_bind_1}' )
X_H_2 = 0.2

E_bind_1 = 
print(f'The bindind energy for X_H = {X_H_2} is E_bind={E_bind_2}' )

X_H_3 = 0.1

E_bind_3 = 
print(f'The bindind energy for X_H = {X_H_3} is E_bind={E_bind_3}' )


<details>
<summary> Solution </summary>

```python
X_H_1 = 0.3
idx_1 = np.where(primary.x_mass_fraction_H <= X_H_1)[0][0]
print(f'The bindind energy for X_H = {X_H} is E_bind={calculate_binding_energy(primary)[idx_1]}' )
X_H_2 = 0.2
idx_2 = np.where(primary.x_mass_fraction_H <= X_H_2)[0][0]
print(f'The bindind energy for X_H = {X_H_2} is E_bind={calculate_binding_energy(primary)[idx_2]}' )

X_H_3 = 0.1
idx_3 = np.where(primary.x_mass_fraction_H <= X_H_3)[0][0]
print(f'The bindind energy for X_H = {X_H_3} is E_bind={calculate_binding_energy(primary)[idx_3]}' )

````

Now that we saw how the binding energy depends on the hydrogen fraction the next step is to calculate how this boundary will affect the post CE orbital seperation and thus the outcome of the CE. 

The fist step is relate the binding energy $E_{bind}$ to the orbital period energy $ E_{orb}$. As the $\alpha_{CE} - \lambda$ prescription this two energy are related by another free parameter $a_{CE}$: $E_{bind} = a_{CE} \, \Delta E_{orb}$ 

For now we will set $a_{CE} = 1$.


### Exercise 5 

Fill in the missing pieces of the function `post_CE_seperation` that given donor mass m1 [$M_\odot$], companion mass m2 [$M_\odot$], donor core mass mc1 [$M_\odot$], initial separation separation_i [$R_\odot$], and E_bind [erg/$M_\odot$], computes separation_postCEE.




<details>
<summary>Hint 1</summary>

The change in orbital energy during CE evolution is:  

$$
\Delta E_{\rm orb} = E_{\rm orb}^{f} - E_{\rm orb}^{i} 
= - \frac{G M_{\rm core} M_1}{2 a_f} + \frac{G M_{\rm donor} M_1}{2 a_0}
$$



In [ ]:
"""
Write the code here.

"""
def post_CE_seperation(m1,m2,mc1,separation_i,binary,E_bind,alpha_CE =1):


    eorb_i =
    
    eorb_postCEE = eorb_i + E_bind/alpha_CE
    
    separation_postCEE = 
    
    return separation_postCEE

<details>
<summary>Solution</summary>

```python
def post_CE_seperation(m1,m2,mc1,separation_i,binary,E_bind,alpha_CE =1):


    eorb_i = (-0.5 * const.standard_cgrav * m1 * const.Msun
              * m2 * const.Msun / (separation_i*const.Rsun))
    
    eorb_postCEE = eorb_i + E_bind/alpha_CE
    
    separation_postCEE = (-0.5 * const.standard_cgrav * mc1 * const.Msun
                          * m2 * const.Msun / eorb_postCEE)/const.Rsun
    return separation_postCEE

```

## Exercise 6 

We need to compute separation_postCEE for X_H = 0.3, 0.2, 0.1. To do that we need to extract from the binary the right quantities like the core mass of the donor without given the boundary definition, as well as the mass of the secondary and primary at the onset of CE



<details>
<summary>Hint</summary>

You need to access the mass and the obrital period at the onset of CE, which is when we stop the MESA run so they are going to be the last element in the binary history array. 

The keys that the binary DataFrame has are : ['model_number', 'age', 'star_1_mass', 'star_2_mass', 'period_days',
       'binary_separation', 'lg_system_mdot_1', 'lg_system_mdot_2',
       'lg_wind_mdot_1', 'lg_wind_mdot_2', 'lg_mstar_dot_1', 'lg_mstar_dot_2',
       'lg_mtransfer_rate', 'xfer_fraction', 'rl_relative_overflow_1',
       'rl_relative_overflow_2', 'trap_radius', 'acc_radius', 't_sync_rad_1',
       't_sync_conv_1', 't_sync_rad_2', 't_sync_conv_2']


Useful tips:
- Use the index from the above exercise to extract mc1 for each X_H threshold: `primary.mass[idx]`
- Use other quantities you calculated before like the binding_energy for the same idx

In [ ]:
"""
Write the code here.

"""

m1 = 
m2 = 
binary_separation_i = 

X_H_1 = 0.3


post_CE_seperation_1 = 
print(f'The bindind energy for X_H = {X_H_1} is E_bind={E_bind_1} and the post_CE_seperation is {post_CE_seperation_1}' )

X_H_2 = 0.2


post_CE_seperation_2 = 
print(f'The bindind energy for X_H = {X_H_2} is E_bind={E_bind_2} and the post_CE_seperation is {post_CE_seperation_2}' )

X_H_3 = 0.1


post_CE_seperation_3 = 
print(f'The bindind energy for X_H = {X_H_3} is E_bind={E_bind_3} and the post_CE_seperation is {post_CE_seperation_3}' )


<details>
<summary>Solution</summary>

```python
m1 = binary.star_1_mass[len(binary)-1]
m2 = binary.star_2_mass[len(binary)-1]
binary_separation = binary.binary_separation[len(binary)-1]

X_H_1 = 0.3
idx_1 = np.where(primary.x_mass_fraction_H <= X_H_1)[0][0]
mc1_1 = primary.mass[idx_1]
E_bind_1 = calculate_binding_energy(primary)[idx_1]
post_CE_seperation_1 = post_CE_seperation(m1,m2,mc1_1,binary_separation,binary,E_bind_1,alpha_CE =1)

print(f'The bindind energy for X_H = {X_H_1} is E_bind={E_bind_1} and the post_CE_seperation is {post_CE_seperation_1}' )

X_H_2 = 0.2
idx_2 = np.where(primary.x_mass_fraction_H <= X_H_2)[0][0]
mc1_2 = primary.mass[idx_2]
E_bind_2 = calculate_binding_energy(primary)[idx_2]
post_CE_seperation_2 = post_CE_seperation(m1,m2,mc1_2,binary_separation,binary,E_bind_2,alpha_CE =1)
print(f'The bindind energy for X_H = {X_H_2} is E_bind={calculate_binding_energy(primary)[idx_2]} and the post_CE_seperation is {post_CE_seperation_2}' )

X_H_3 = 0.1
idx_3 = np.where(primary.x_mass_fraction_H <= X_H_3)[0][0]
mc1_3 = primary.mass[idx_3]
E_bind_3 = calculate_binding_energy(primary)[idx_3]
post_CE_seperation_3 = post_CE_seperation(m1,m2,mc1_3,binary_separation,binary,E_bind_3,alpha_CE =1)
print(f'The bindind energy for X_H = {X_H_3} is E_bind={calculate_binding_energy(primary)[idx_3]} and the post_CE_seperation is {post_CE_seperation_2}' )

```

The binary will survive the CE if the both stars can fit within their RL, in the new binary configuration. Run the next block of code to see the outcome of the CE for each of core-envelope definitions. Depending on the binary all outcomes might merge of survive the CE phase.

In [ ]:
def ce_outcome(primary, secondary,binary,X_thr, alpha_CE=1.0 ):
    
    idx = np.where(primary.x_mass_fraction_H <= X_thr)[0][0]

    mc1   = float(primary.mass.iloc[idx])
    R1    = float(primary.radius.iloc[idx])      
    R2    = float(secondary.radius.iloc[0])      
    Ebind = float(calculate_binding_energy(primary)[idx])
    initial_seperation = binary.binary_separation[len(binary)-1]
    final_seperation   = post_CE_seperation(m1, m2, mc1, initial_seperation,binary, Ebind, alpha_CE=1.0)

    RL1 = cf.roche_lobe_radius(mc1, m2, final_seperation)
    RL2 = cf.roche_lobe_radius(m2, mc1, final_seperation)
    survive = (R1 < RL1) and (R2 < RL2)
    return survive


for X_thr in [0.3, 0.2, 0.1,0.01]:
    survival =  ce_outcome(primary, secondary,binary,X_thr)
    outcome = "SURVIVE (detached)" if survival else "MERGE"
    print(f"X_H = {X_thr:0.2f}: -> {outcome}")

The boundary definition is not the only free parameter that can determine the outcome, there the parameter `alpha_CE` that so far we kept fixed. Run the code bellow and see how the outcome of the CE changes. 

In [ ]:
XH_values   = [0.3, 0.2, 0.1]
alpha_values = [0.2, 0.5, 1.0, 2.0, 3.0]

for alpha_CE in alpha_values:
    for X_thr in XH_values:
        survival =  ce_outcome(primary, secondary,binary,X_thr,alpha_CE = alpha_CE)
        outcome = "SURVIVE (detached)" if survival else "MERGE"
        print(f"X_H = {X_thr:0.1f} and alpha_CE = {alpha_CE}: -> {outcome}")


### The λ parameter

---

In the α<sub>CE</sub>–λ scheme, the binding energy is often expressed as:  

$$
E_{\rm bind} = - \frac{G M_{\rm donor} \, M_{\rm env}}{\lambda \, R_{\rm donor}}
$$


Where the structure parameter $\lambda$ depends on the stellar density of the star. The value of this parameter depends on the evolutionary state and structure of the donor. Generally, in BPS codes the $\lambda$ coeffiecent is estimated using tables or fitting formulas of precomputed values for a given mass from the literature ( Dewi & Tauris, 2000, 2001; Xu & Li, 2010; Loveridge et al., 2011 , Claes et al. 2014). These predifined values are very sensitive to the definition of the core mass coordinate but also depend on the prior evolution that the star had. 

---
In **POSYDON** we don't have to rely on approximations or fitted formulas, since we have access to the final stellar profiles right before the binary enters CE.  
This allows us to:

1. Calculate λ directly from the detailed MESA profiles for each grid.
2. Calculate λ for any arbitrary binary by using the initial-final interpolator capabilities of POSYDON.
3. POSYDON also has different λ values for 3 different co-envelope boundary definitions: $X_H = 0.01,\; 0.1,\; \text{or}\; 0.3$
4. Calculate λ on-the-fly for a given donor in our grids, using a different binding energy definition.    


---

### What happends after a successful CE ejection ? 

If the binary survives the CE, **POSYDON** matches the donor star to stripped He star that has only X_H = 0.01 left in it's envelope. The question here is how did the donor lost the reminder of it's envelope ? 

There are 3 possible options: 
1. The envelope was lost through another stable mass transfer episode after the ejection. 
2. The envelope was lost due to wind. 
3. Nothing happens 


# CE step in the POSYDON flow

POSYDON has a dedicated step for handling the evolution of binaries that enter a common-envelope phase. The CE step performs three main tasks:


- **Step 1. Implements the CE prescription:** It extracts the necessary binary and stellar properties of the binary at the onset of the CE and calculates post-CE orbital separation.
  
- **Step 2. Determines the CE outcome:** It accesses whether the post-CE binary properties will result in a detached binary or merger and calculates the binary properties of the post-CE system. 

  
- **Step 3. Directs the binary to the next step in the flow:** If the envelope is successfully ejected, the binary continues to the detached step, with the donor matched to a stripped-He star. If the envelope cannot be ejected, the binary is directed to the merged step.




# CE options in POSYDON population runs

Bellow is a screenshot of the .ini listing all the possible options for the CE step. 


![The CE option running a population](./ini_CE.png)


## Exercise 7

In this exercise we will run the binary from the grids and change the CE step. You can choose based on the previous test a set of parameters that result in a succeful ejection or merger. Run the next block of code and then edit STAR1,STAR2, BINARY orbital and give the initial masses and orbital period of the binary selected from the grid. 

In [ ]:
from posydon.binary_evol.binarystar import BinaryStar, SingleStar
from posydon.binary_evol.simulationproperties import SimulationProperties
from posydon.popsyn.io import simprop_kwargs_from_ini

sim_kwargs = simprop_kwargs_from_ini('./population_params.ini', verbose=True)
metallicity = {'metallicity':1}

sim_kwargs['step_HMS_HMS'][1].update(metallicity)
sim_kwargs['step_CO_HeMS'][1].update(metallicity)
sim_kwargs['step_CO_HMS_RLO'][1].update(metallicity)
sim_kwargs['step_CO_HeMS_RLO'][1].update(metallicity)
sim_kwargs['step_detached'][1].update(metallicity)
sim_kwargs['step_disrupted'][1].update(metallicity)
sim_kwargs['step_merged'][1].update(metallicity)
sim_kwargs['step_initially_single'][1].update(metallicity)
sim_kwargs['step_CE'][1].update(metallicity)
sim_kwargs['step_detached'][1].update(metallicity)
sim_kwargs['step_HMS_HMS'][1].update({'track_interpolation':False})
sim_pop = SimulationProperties(**sim_kwargs)

# Put your own values for the common_envelope_efficiency, core_definition_H_fraction
sim_kwargs['step_CE'][1].update({'verbose':True,'common_envelope_efficiency' : ,'core_definition_H_fraction': ,'common_envelope_option_after_succ_CEE':,})

sim_pop.load_steps()

Reading inifile: 
	/blue/jeffrey.andrews/kasdaglie/POSYDON-class-materials-2026/2_Tuesday_25_8_26/CE_1/population_params.ini


In [ ]:
#Fill in the mass of the primary (STAR1), the secondary (STAR2) and the orbital period
STAR1 = SingleStar(**{'mass':,
                      'state': 'H-rich_Core_H_burning'})
STAR2 = SingleStar(**{'mass': ,
                      'state': 'H-rich_Core_H_burning'})

BINARY = BinaryStar(STAR1, STAR2,
                    **{'time': 0.0, 'state': 'detached', 'event': 'ZAMS', 'orbital_period': , 'eccentricity': 0.0},properties = sim_pop)

In [ ]:
# Run this code to evolve the binary and save it as pandas DataFrame. 
BINARY.evolve()
History = BINARY.to_df(**{'extra_columns':{'step_names':'str'}})


Now you can inspect the evolution in the binary in POSYDON and what happen to the binary post-CE 

In [ ]:

History[['state','step_names','orbital_period','time',
         'S1_mass','S1_state','S2_mass','S2_state','S2_log_R',
         'mass_transfer_case','S2_state','S2_log_R','S2_mass', 'event', 'step_names' ,'lg_mtransfer_rate','S2_surf_avg_omega']]
         